# Chapter 1 — Build It Yourself: The Bigram Language Model

Welcome! This notebook is the hands-on heart of [Chapter 1](../README.md). It is
**not** a demo to click through — *you* write the code, and I'll check your work as
you go, like a tutor sitting next to you.

**Three kinds of cell:**
- **✍️ Your turn** — you write the missing line(s). The cell gives you a *hint*; if
  you get stuck, click **"Stuck? Show the answer"** right above it. Typing it yourself
  (not copy-pasting) is what makes it stick.
- **📖 Study & run** — the two hardest cells are shown in full with a line-by-line
  explanation. Read them closely, then run.
- **▶️ Run this / ▶️ Check your work** — already complete; just run. The check cells
  tell you instantly whether you nailed it (✅) or need a nudge (❌).

Work top to bottom — later cells depend on earlier ones. Let's go. 🚀

## Step 1 — Setup & load the data  ▶️ Run this

We import PyTorch and load all 32,033 names. Nothing to write — just run it.

In [ ]:
from pathlib import Path
import torch
import torch.nn.functional as F

DATA = next(p for p in [Path("../data/names.txt"),
                        Path("data/names.txt"),
                        Path("chapters/01-bigram/data/names.txt")] if p.exists())
words = [w.strip() for w in DATA.read_text().splitlines() if w.strip()]
print(f"{len(words)} names loaded |", words[:5])

## Step 2 — Build the vocabulary  ✍️ Your turn

Map each character to an integer **id** and back. Letters get ids `1..26`; the special
boundary token `'.'` gets id `0`.

**Your task:** build `stoi` (char → id), `itos` (id → char), and `V` (how many tokens).

<details><summary>Stuck? Show the answer</summary>

```python
chars = sorted(set("".join(words)))
stoi = {c: i + 1 for i, c in enumerate(chars)}
stoi["."] = 0
itos = {i: c for c, i in stoi.items()}
V = len(stoi)
```
</details>

In [ ]:
chars = sorted(set("".join(words)))
# ✍️ Your turn (your very first one!). Build three things:
#   1) stoi  — map each char in `chars` to a number 1, 2, 3, …
#              (a dict comprehension over enumerate(chars) is the tidy way)
#   2) itos  — the reverse of stoi (id -> char)
#   3) V     — how many tokens there are
stoi = {}              # ✍️ build the char->id dict here
stoi["."] = 0          # ← then this adds the boundary token (id 0). Done for you!
itos = {}              # ✍️ make this the reverse of stoi
V = 0                  # ✍️ the number of tokens
print(V, stoi)

In [ ]:
# ▶️ Check your work
try:
    assert V == 27, "V should be 27 (26 letters + the '.' token)"
    assert stoi.get(".") == 0, "the '.' token must map to 0"
    assert stoi.get("a") == 1 and stoi.get("z") == 26, "letters should map to 1..26 in order"
    assert itos.get(5) == "e", "itos should be the exact reverse of stoi"
    print("✅ Vocabulary perfect — 27 tokens, '.'→0, a→1 … z→26.")
except NameError:
    print("❌ stoi / itos / V aren't defined yet — fill in and run the cell above first.")
except AssertionError as e:
    print("❌ Not quite:", e)

## Step 3 — A helper for bigrams  ✍️ Your turn

A **bigram** is a pair of neighbouring characters. We pad each name with `'.'` at both
ends, then read off every adjacent pair.

**Your task:** return all adjacent pairs of `chs`. *(Hint: `zip` the list with itself
shifted by one.)*

<details><summary>Stuck? Show the answer</summary>

```python
return list(zip(chs, chs[1:]))
```
</details>

In [ ]:
def bigrams(word):
    chs = ["."] + list(word) + ["."]
    # ✍️ return a list of every adjacent (char, char) pair in chs
    #    (think: pair each item with the one that comes right after it)
    return []

print(bigrams("emma"))

In [ ]:
# ▶️ Check your work
try:
    assert bigrams("emma") == [('.','e'),('e','m'),('m','m'),('m','a'),('a','.')], \
        "expected the 5 padded pairs for 'emma' (don't forget the '.' at both ends)"
    print("✅ Nice — your bigrams wrap the name in '.' and pair up neighbours correctly.")
except NameError:
    print("❌ Define bigrams() in the cell above first.")
except AssertionError as e:
    print("❌ Not quite:", e)

## Step 4 — Count every bigram into a 27×27 grid  ✍️ Your turn

`N[i, j]` will hold *"how many times did character `j` follow character `i`?"*

**Your task:** inside the loop, add 1 to the cell at row = id of `a`, col = id of `b`.

<details><summary>Stuck? Show the answer</summary>

```python
N[stoi[a], stoi[b]] += 1
```
</details>

In [ ]:
N = torch.zeros((V, V), dtype=torch.int32)
for w in words:
    for a, b in bigrams(w):
        # ✍️ add 1 to the cell at row = id of a, col = id of b
        pass

print("N shape:", tuple(N.shape), "| total bigrams:", int(N.sum()))
print("q->u:", int(N[stoi['q'], stoi['u']]), "| q->z:", int(N[stoi['q'], stoi['z']]))

In [ ]:
# ▶️ Check your work
try:
    assert tuple(N.shape) == (27, 27), "N should be 27x27"
    assert int(N.sum()) == 228146, "N should total 228,146 bigrams — is the += inside BOTH loops?"
    assert int(N[stoi['q'], stoi['u']]) == 206, "q→u should be 206 — check the order: row = first char, col = second"
    print("✅ Counts look right — 228,146 bigrams, and q→u = 206 (almost every q is followed by u!).")
except NameError:
    print("❌ Build N in the cell above first.")
except AssertionError as e:
    print("❌ Not quite:", e)

## Step 4b — See the model  ▶️ Run this

Bright first column = letters names end on; bright first row = letters names start
with; spot `q`→`u`. This picture **is** the model.

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(14, 14))
plt.imshow(N, cmap="Blues")
for i in range(V):
    for j in range(V):
        plt.text(j, i, itos[i] + itos[j], ha="center", va="bottom", color="gray", fontsize=6)
        plt.text(j, i, int(N[i, j]), ha="center", va="top", color="gray", fontsize=6)
plt.axis("off"); plt.title("Bigram counts"); plt.show()

## Step 5 — Turn counts into probabilities  📖 Study this one, then run

This is the chapter's trickiest line, so it's shown **in full** — read it closely.
Each row of `N` should become 27 probabilities that **sum to 1**: add `1` to every
cell (smoothing, so nothing is ever 0), then divide each row by its total.

```python
P = (N + 1).float()
P /= P.sum(dim=1, keepdim=True)
```

⚠️ **Why `keepdim=True`?** `P.sum(dim=1, keepdim=True)` has shape `(27, 1)` — a
*column* — so dividing **broadcasts** correctly: row `i` ÷ row `i`'s total. Drop
`keepdim=True` and the shape is `(27,)`, which lines up as a *row* and divides the
**wrong way**, silently. The check below confirms the shape is `[27, 1]`. Run it:

In [ ]:
P = (N + 1).float()
P /= P.sum(dim=1, keepdim=True)

print("row 0 sums to:", float(P[0].sum()))
print("sum shape (must be [27, 1]):", list(P.sum(dim=1, keepdim=True).shape))

In [ ]:
# ▶️ Check your work
try:
    assert list(P.sum(dim=1, keepdim=True).shape) == [27, 1], "row-sums must be shape [27, 1] — use keepdim=True"
    assert abs(float(P[0].sum()) - 1.0) < 1e-5, "each row must sum to 1"
    assert torch.allclose(P.sum(1), torch.ones(27), atol=1e-4), "every row should sum to 1, not just row 0"
    print("✅ Every row is a proper probability distribution. The keepdim trap: handled!")
except NameError:
    print("❌ Run the cell above first.")
except AssertionError as e:
    print("❌ Not quite:", e)

## Step 6 — Generate names!  ✍️ Your turn

Start at `'.'`, look up its row, **sample** the next character with `torch.multinomial`
(a weighted die), move to that row, repeat until you draw `'.'` again.

**Your task:** write the one line that samples the next index `ix` from `p`.

<details><summary>Stuck? Show the answer</summary>

```python
ix = torch.multinomial(p, num_samples=1, generator=g).item()
```
</details>

In [ ]:
def sample_name(P, g):
    ix, out = 0, []
    while True:
        p = P[ix]
        # ✍️ sample the next index from p — use torch.multinomial(...) and .item()
        ix = 0   # ✍️ REPLACE this whole line: sample the next index from p
        if ix == 0:
            break
        out.append(itos[ix])
    return "".join(out)

g = torch.Generator().manual_seed(2147483647)
for _ in range(10):
    print(" ", sample_name(P, g))

In [ ]:
# ▶️ Check your work
try:
    _g = torch.Generator().manual_seed(2147483647)
    _first = sample_name(P, _g)
    assert _first == "cexze", f"with this seed the first name should be 'cexze', but you got {_first!r} — check your multinomial line"
    print("✅ Sampling works — your first name is 'cexze', exactly like the lesson. You're generating language!")
except (NameError, RuntimeError):
    print("❌ Finish sample_name() in the cell above first (it should sample, not return 0).")
except AssertionError as e:
    print("❌ Not quite:", e)

## Step 7 — Measure quality: the loss  ✍️ Your turn

The **average negative log-likelihood**: add up the log-probability of every real
bigram, negate, average. The clueless baseline is `log(27) ≈ 3.30`.

🔮 **Predict first:** will our model beat 3.30, and by how much?

**Your task:** inside the loop, add `math.log` of this bigram's probability to
`log_likelihood`, and increment `n`.

<details><summary>Stuck? Show the answer</summary>

```python
log_likelihood += math.log(P[stoi[a], stoi[b]])
n += 1
```
</details>

In [ ]:
import math
log_likelihood, n = 0.0, 0
for w in words:
    for a, b in bigrams(w):
        # ✍️ add math.log of P[id of a, id of b] to log_likelihood, then n += 1
        pass

nll = -log_likelihood / n if n else float("nan")
print(f"loss = {nll:.4f}   (clueless baseline = {math.log(27):.4f})")

In [ ]:
# ▶️ Check your work
try:
    assert round(nll, 2) == 2.45, f"expected about 2.45, but got {nll:.4f}"
    print(f"✅ Loss = {nll:.4f} — well below the 3.30 baseline, so the model genuinely learned. Remember it for Step 9!")
except NameError:
    print("❌ Compute nll in the cell above first.")
except AssertionError as e:
    print("❌ Not quite:", e)

## Step 8 — Same model as a neural net: build the dataset  ✍️ Your turn

Flatten every bigram into `(input id, target id)` pairs.

**Your task:** append the id of `a` to `xs`, and the id of `b` to `ys`.

<details><summary>Stuck? Show the answer</summary>

```python
xs.append(stoi[a]); ys.append(stoi[b])
```
</details>

In [ ]:
xs, ys = [], []
for w in words:
    for a, b in bigrams(w):
        # ✍️ append id of a to xs, and id of b to ys
        pass

xs, ys = torch.tensor(xs), torch.tensor(ys)
print("examples:", xs.nelement())

In [ ]:
# ▶️ Check your work
try:
    assert xs.nelement() == 228146 and ys.nelement() == 228146, "should be 228,146 examples each"
    assert int(xs[0]) == 0 and int(ys[0]) == stoi['e'], "the very first example should be ('.', 'e') from the name 'emma'"
    print("✅ Dataset ready — 228,146 (input → target) pairs, starting with ('.', 'e').")
except (NameError, IndexError):
    print("❌ Build xs and ys in the cell above first.")
except AssertionError as e:
    print("❌ Not quite:", e)

## Step 9 — Train it: forward → backward → update  📖 Study this one, then run

This is the most important cell in the chapter, so it's shown **in full**. The loop
repeats three steps: **forward** (predict + measure loss), **backward**
(`loss.backward()` finds the gradients), **update** (nudge `W` downhill).

**The `loss` line, unpacked** (the densest line in the chapter):
- `probs` has one row per example; row `i` = the model's 27 predicted probabilities.
- `probs[torch.arange(num), ys]` is **paired indexing**: `torch.arange(num)` is
  `[0,1,…,num-1]`, and pairing it with `ys` plucks, from each row, the probability the
  model gave to the **correct** next character — for all 228k examples at once. (It's
  Step 7's `P[id, id]`, vectorised.) Note `num` = number of examples, *not* the 27×27
  grid `N` from Step 4.
- `.log().mean()` logs each and averages; the leading `-` makes it the average
  **negative** log-likelihood — the same loss as Step 7.
- `+ 0.01 * (W**2).mean()` is a small regularization nudge.

Watch the loss fall from ~3.77 toward the same ~2.45 you got by counting, then run it:

In [ ]:
num = xs.nelement()
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((V, V), generator=g, requires_grad=True)   # random start; track grads

for step_i in range(100):
    # forward
    xenc = F.one_hot(xs, num_classes=V).float()            # (num, 27) one-hots
    probs = F.softmax(xenc @ W, dim=1)                     # (num, 27) predicted probabilities
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01 * (W**2).mean()
    # backward
    W.grad = None
    loss.backward()                                        # autograd! (demystified in Ch 2)
    # update
    W.data += -50 * W.grad                                 # step downhill (learning rate 50)
    if step_i % 20 == 0:
        print(f"step {step_i:3d} | loss {loss.item():.4f}")

In [ ]:
# ▶️ Check your work
try:
    assert W.grad is not None, "W.grad is None — did loss.backward() run?"
    assert loss.item() < 2.6, f"after 100 steps the loss should be under 2.6, but it's {loss.item():.4f}"
    print(f"✅ It trained! Final loss {loss.item():.4f}, closing in on the counting model's 2.45 — the SAME model, found by learning. 🤯")
except NameError:
    print("❌ Run the training cell above first.")
except AssertionError as e:
    print("❌ Not quite:", e)

## Step 10 — Generate from your trained network  ✍️ Your turn

Same idea as Step 6, but the distribution now comes from the network
(`softmax(one_hot(ix) @ W)`).

**Your task:** the same sampling line as Step 6.

<details><summary>Stuck? Show the answer</summary>

```python
ix = torch.multinomial(p, num_samples=1, generator=g).item()
```
</details>

In [ ]:
@torch.no_grad()
def sample_nn(W, g):
    ix, out = 0, []
    while True:
        p = F.softmax(F.one_hot(torch.tensor([ix]), num_classes=V).float() @ W, dim=1)
        # ✍️ sample the next index from p (same line as Step 6)
        ix = 0   # ✍️ REPLACE this whole line: sample the next index from p
        if ix == 0:
            break
        out.append(itos[ix])
    return "".join(out)

g = torch.Generator().manual_seed(2147483647)
for _ in range(10):
    print(" ", sample_nn(W, g))

In [ ]:
# ▶️ Check your work
try:
    _g = torch.Generator().manual_seed(2147483647)
    _nm = sample_nn(W, _g)
    assert isinstance(_nm, str) and len(_nm) >= 1 and all(c in itos.values() for c in _nm), "should return a non-empty name of valid characters"
    print(f"✅ Your neural net speaks! It generated {_nm!r}. Same kind of names as counting — because it IS the same model.")
except (NameError, RuntimeError):
    print("❌ Finish sample_nn() in the cell above first (it should sample, not return 0).")
except AssertionError as e:
    print("❌ Not quite:", e)

## 🎓 You built a language model — twice.

By counting *and* by gradient descent, and they agree. That second loop — forward,
backward, update — is exactly what scales up to ChatGPT.

**Next:**
- Try the [exercises](../exercises/) — build a **trigram** model (it's better!).
- Build the [mini-project](../project/), *The Name Forge*.
- Then [Chapter 2 — Micrograd](../../02-micrograd/), where we open up `loss.backward()`
  and build backprop ourselves.

Brilliant work. See you in Chapter 2. 👋